In [ ]:
import os, glob, json, time
import numpy as np, pandas as pd, torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
print("GPU:", torch.cuda.get_device_name(0), flush=True)
pairs = pd.read_parquet(glob.glob("/kaggle/input/**/eval_pairs_mixed.parquet", recursive=True)[0])
tx = pd.read_parquet(glob.glob("/kaggle/input/**/human_texts.parquet", recursive=True)[0], columns=["id","text"])
tmap = dict(zip(tx["id"].tolist(), tx["text"].tolist()))
L=[str(tmap.get(int(i),"")) for i in pairs["id1"]]; R=[str(tmap.get(int(i),"")) for i in pairs["id2"]]
print(f"пар {len(pairs):,}, доля+ {pairs['target'].mean():.3f}", flush=True)
dirs={}
for p in glob.glob("/kaggle/input/**/inference_config.json", recursive=True):
    d=os.path.dirname(p)
    if "checkpoint" not in d: dirs[os.path.basename(d)]=d
print("модели:", sorted(dirs), flush=True)
order=np.argsort([len(a)+len(b) for a,b in zip(L,R)])
os.makedirs("/kaggle/working/scores", exist_ok=True)
for name,d in sorted(dirs.items()):
    ml=int(json.load(open(d+"/inference_config.json"))["max_length"])
    tok=AutoTokenizer.from_pretrained(d, local_files_only=True)
    mod=AutoModelForSequenceClassification.from_pretrained(d, local_files_only=True, dtype=torch.float16).cuda().eval()
    out=np.empty(len(L), dtype=np.float32); t=time.perf_counter()
    with torch.inference_mode():
        for i in range(0, len(order), 512):
            rows=order[i:i+512]
            e=tok([L[j] for j in rows],[R[j] for j in rows],padding=True,truncation=True,
                  max_length=ml,pad_to_multiple_of=8,return_tensors="pt").to("cuda")
            out[rows]=mod(**e).logits.squeeze(-1).float().cpu().numpy()
    np.save(f"/kaggle/working/scores/{name}.npy", out)
    print(f"{name}: {time.perf_counter()-t:.0f}с", flush=True)
    del mod; torch.cuda.empty_cache()
pairs.to_parquet("/kaggle/working/scores/eval_pairs_mixed.parquet", index=False)
print("готово", flush=True)
